Summary
Goal
Find Chinese-English idiom pairs that share the same figurative meanings but use different entities, revealing how cultures express similar concepts differently.

What Was Built
Main Script (cross_lingual_same_meaning_diff_entity.py):

embed command: Computes one embedding per figurative meaning (not per idiom), handles nested lists in data, saves efficiently as compressed .npz + metadata JSON
pairs command: Computes cross-lingual similarity, finds best matches, analyzes entity differences
Helper Script (reanalyze_pairs.py):

Re-runs analysis on existing pairs without recomputing embeddings
Key Features
Separate embeddings per figurative meaning for precise matching
Efficient storage (~10-20x smaller than JSON)
Saves all examples where both languages have entities (fixed from top-20 limit)
Output Files
figurative_embeddings.npz / _meta.json: Embeddings per language
cross_lingual_pairs.jsonl: Matched idiom pairs with similarity scores
entity_analysis.json: Statistics and all examples with entity differences
Example Finding
A Chinese idiom about "tigers" and an English idiom about "lions" might share the meaning "powerful/dangerous" — same concept, different cultural entities.

In [ ]:
# Chinese idioms
python3 /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/src/culture/analysis/cross_lingual_same_meaning_diff_entity.py embed \
  --input /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/zh/idioms_merged_llm_formatted.jsonl \
  --embedding_output /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/zh/figurative_embeddings

# English idioms
python3 /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/src/culture/analysis/cross_lingual_same_meaning_diff_entity.py embed \
  --input /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/en/idioms_merged_llm_formatted.jsonl \
  --embedding_output /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/en/figurative_embeddings

In [ ]:
python3 /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/src/culture/analysis/cross_lingual_same_meaning_diff_entity.py pairs \
  --zh_embeddings /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/zh/figurative_embeddings \
  --en_embeddings /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/en/figurative_embeddings \
  --pairs_output /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/cross_lingual_pairs.jsonl \
  --analysis_output /home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/entity_analysis.json \
  --threshold 0.7



## Same Meaning Across Different Entities (Intra-lingual Clustering)

### Goal
Group idioms sharing the same figurative meaning **within** each language, then bridge Chinese and English clusters using cross-lingual pairs as anchors -- producing bilingual semantic clusters.

### What Was Built
**Script** (`intra_lingual_idiom_clusters.py`):

1. Loads pre-computed figurative-meaning embeddings (from step above) for both languages
2. Computes **intra-lingual** similarity pairs within Chinese and within English (above a cosine threshold)
3. Uses **cross-lingual pairs as anchors** to seed bilingual clusters
4. Expands each cluster by finding idioms whose embeddings are similar to the anchor's **specific matched meaning** (not any meaning), preventing false groupings from polysemous idioms
5. Formats and saves bilingual clusters with shared meaning, entities, and idiom lists

### Key Design Decisions
- **Meaning-specific expansion**: only adds idioms similar to the anchor pair's matched meaning, not any of the idiom's meanings
- **Strict default threshold** (`0.95`): tighter than the cross-lingual step (`0.7`) to keep intra-lingual clusters precise
- **Parallel processing**: optional `--parallel` flag with configurable `--n_workers` for large datasets
- **Caching**: skips recomputation of intra-lingual pairs if cached files exist (override with `--recompute_intra`)

### Output Files
| File | Description |
|------|-------------|
| `bilingual_clusters.jsonl` / `.json` | Clusters with shared meaning, anchor pair, Chinese idioms, and English idioms |
| `zh_intra_lingual_pairs.jsonl` | Within-Chinese similar idiom pairs |
| `en_intra_lingual_pairs.jsonl` | Within-English similar idiom pairs |

Each cluster contains: `shared_meaning`, `anchor_zh_idiom`, `anchor_en_idiom`, `zh_idioms` (with entities/figurative meanings), `en_idioms` (with entities/figurative meanings).